# LeetCode 1784 — Check if Binary String Has at Most One Segment of Ones bằng Python

Link đề: https://leetcode.com/problems/check-if-binary-string-has-at-most-one-segment-of-ones/

Độ khó: **Easy** · Chủ đề: String

Chữ ký hàm cần viết (LeetCode, Python 3):

```python
class Solution:
    def checkOnesSegment(self, s: str) -> bool:

```

**Mục tiêu học:**

1. Hiểu khái niệm **đoạn liên tiếp (contiguous segment)** trong một chuỗi.
2. Biết duyệt chuỗi từng ký tự, so sánh ký tự hiện tại với ký tự trước đó.
3. Biết đếm "điểm bắt đầu đoạn" — kỹ thuật dùng lại cho rất nhiều bài chuỗi/mảng.
4. Nhận ra **insight**: vì `s[0] == '1'`, có hai đoạn 1 khi và chỉ khi chuỗi chứa `"01"`.
5. So sánh 4 cách giải: đếm đoạn, `split`, cờ trạng thái (flag), tìm `"01"`.

## 1. Bài toán yêu cầu gì?

Cho chuỗi nhị phân `s` (chỉ gồm `'0'` và `'1'`), **không có số 0 ở đầu** (tức `s[0] == '1'`).

Trả về `True` nếu `s` chứa **nhiều nhất một** đoạn liên tiếp toàn số `1`; ngược lại trả về `False`.

"Đoạn liên tiếp các số 1" = một khúc chuỗi gồm các `'1'` đứng sát nhau, hai đầu bị chặn bởi `'0'` hoặc biên chuỗi.

| `s` | Các đoạn 1 | Kết quả |
|---|---|:---:|
| `"1001"` | `"1"`, `"1"` → 2 đoạn | `False` |
| `"110"` | `"11"` → 1 đoạn | `True` |
| `"1"` | `"1"` → 1 đoạn | `True` |
| `"111000"` | `"111"` → 1 đoạn | `True` |
| `"1101"` | `"11"`, `"1"` → 2 đoạn | `False` |

**Ràng buộc:** `1 <= len(s) <= 100`, `s[i]` là `'0'` hoặc `'1'`, `s[0] == '1'`.

Chú ý: vì `s[0] == '1'` nên luôn có **ít nhất một** đoạn. Câu hỏi thực chất là: *có đúng 1 đoạn hay có từ 2 đoạn trở lên?*

In [1]:
# Các ví dụ của đề, lưu thành list để dùng lại về sau
examples = [
    ("1001", False),
    ("110", True),
]
for s, expected in examples:
    print(f"s = {s!r:8} -> kỳ vọng {expected}")

s = '1001'   -> kỳ vọng False
s = '110'    -> kỳ vọng True


## 2. Nền tảng Python cần biết

### 2.1. Chuỗi, chỉ số (index) và duyệt từng ký tự

Một chuỗi `str` trong Python giống một dãy ký tự có đánh số từ `0`.

- `s[0]` là ký tự đầu, `s[-1]` là ký tự cuối.
- `len(s)` là số ký tự.
- `for ch in s:` lấy lần lượt từng ký tự.
- `for i in range(len(s)):` lấy lần lượt từng **chỉ số** `0, 1, ..., len(s)-1`.

In [2]:
s = "1001"
print("len:", len(s))
print("s[0] =", s[0], "| s[-1] =", s[-1])

for ch in s:
    print("ký tự:", ch)

for i in range(len(s)):
    print("chỉ số", i, "->", s[i])

len: 4
s[0] = 1 | s[-1] = 1
ký tự: 1
ký tự: 0
ký tự: 0
ký tự: 1
chỉ số 0 -> 1
chỉ số 1 -> 0
chỉ số 2 -> 0
chỉ số 3 -> 1


### 2.2. `enumerate` — lấy cả chỉ số lẫn ký tự

`enumerate(s)` trả về từng cặp `(i, ch)`. Rất tiện khi vừa cần vị trí vừa cần giá trị.

In [3]:
for i, ch in enumerate("110"):
    print(i, ch)

0 1
1 1
2 0


### 2.3. So sánh với ký tự đứng trước: `s[i-1]`

Muốn biết vị trí `i` có phải **điểm bắt đầu một đoạn 1** không, ta hỏi:

> `s[i] == '1'` **và** (ký tự trước là `'0'` hoặc `i == 0`)?

Cẩn thận: khi `i == 0` thì `s[i-1]` là `s[-1]` — **ký tự cuối chuỗi** (Python không báo lỗi!). Đó là lỗi rất hay gặp, nên ta phải tách riêng `i == 0`.

In [4]:
s = "1001"
print("s[-1] khi i = 0:", s[0 - 1], "<- đây là ký tự CUỐI, không phải 'ký tự trước'!")

for i in range(len(s)):
    is_start = s[i] == "1" and (i == 0 or s[i - 1] == "0")
    print(f"i={i}, s[i]={s[i]}, bắt đầu đoạn 1? {is_start}")

s[-1] khi i = 0: 1 <- đây là ký tự CUỐI, không phải 'ký tự trước'!
i=0, s[i]=1, bắt đầu đoạn 1? True
i=1, s[i]=0, bắt đầu đoạn 1? False
i=2, s[i]=0, bắt đầu đoạn 1? False
i=3, s[i]=1, bắt đầu đoạn 1? True


### 2.4. Toán tử `in` với chuỗi con và `str.split`

- `"01" in s` trả về `True` nếu chuỗi con `"01"` xuất hiện **liền nhau** ở đâu đó trong `s`.
- `s.split("0")` cắt chuỗi tại mỗi `'0'` và trả về list các mảnh (có thể có mảnh rỗng `''`).

In [5]:
s = "1001"
print('"01" in s ->', "01" in s)
print('"01" in "110" ->', "01" in "110")

print('"1001".split("0") ->', "1001".split("0"))
print('"110".split("0")  ->', "110".split("0"))
print('"11011".split("0") ->', "11011".split("0"))

"01" in s -> True
"01" in "110" -> False
"1001".split("0") -> ['1', '', '1']
"110".split("0")  -> ['11', '']
"11011".split("0") -> ['11', '11']


Quan sát: `split("0")` sinh ra mảnh rỗng `''` khi có hai số `0` liền nhau hoặc `0` ở cuối. Mảnh **không rỗng** chính là một đoạn 1.

### 2.5. List comprehension và `len`

`[p for p in parts if p]` là cách viết gọn để "giữ lại các phần tử `p` mà `p` không rỗng". Chuỗi rỗng `''` được Python coi là **False** trong `if`.

In [6]:
parts = "100110".split("0")
print("parts:", parts)
non_empty = [p for p in parts if p]
print("mảnh không rỗng:", non_empty, "-> số đoạn 1 =", len(non_empty))
print("bool('') =", bool(""), "| bool('11') =", bool("11"))

parts: ['1', '', '11', '']
mảnh không rỗng: ['1', '11'] -> số đoạn 1 = 2
bool('') = False | bool('11') = True


## 3. Cách trực giác — đếm số đoạn 1 (baseline)

Người mới thường nghĩ: *"Đếm xem có bao nhiêu đoạn 1, rồi so với 1."*

Cách đếm: một đoạn 1 **bắt đầu** tại vị trí `i` khi `s[i] == '1'` và (`i == 0` hoặc `s[i-1] == '0'`). Mỗi đoạn có đúng một điểm bắt đầu, nên **số điểm bắt đầu = số đoạn**.

In [7]:
def count_one_segments(s: str) -> int:
    count = 0
    for i in range(len(s)):
        if s[i] == "1" and (i == 0 or s[i - 1] == "0"):
            count += 1
    return count

def check_ones_segment_count(s: str) -> bool:
    return count_one_segments(s) <= 1

for s in ["1001", "110", "1", "10101", "111000"]:
    print(f"{s!r:10} số đoạn = {count_one_segments(s)} -> {check_ones_segment_count(s)}")

'1001'     số đoạn = 2 -> False
'110'      số đoạn = 1 -> True
'1'        số đoạn = 1 -> True
'10101'    số đoạn = 3 -> False
'111000'   số đoạn = 1 -> True


### 3.1. Baseline có vấn đề gì không?

- Đúng, chạy `O(n)` — với `n <= 100` thì quá đủ.
- Nhưng nó **làm thừa việc**: đếm *tất cả* các đoạn, trong khi ta chỉ cần biết "có đoạn thứ hai hay không". Gặp đoạn thứ hai là có thể dừng ngay.
- Code còn phải cẩn thận biên `i == 0`.

Câu hỏi gợi mở: *Đoạn thứ hai bắt đầu trông như thế nào trong chuỗi?*

## 4. Nút thắt và insight

Hãy nhìn vị trí **bắt đầu của đoạn 1 thứ hai**. Trước nó chắc chắn là một `'0'` (vì đoạn thứ nhất đã kết thúc bằng `0`). Vậy tại đó ta thấy cặp ký tự liền nhau **`"01"`**.

Ngược lại, nếu chuỗi có cặp `"01"` ở đâu đó: ký tự `'1'` trong cặp đó bắt đầu một đoạn 1 **không phải đoạn đầu** (vì đoạn đầu bắt đầu ở `s[0]`, đề đảm bảo `s[0] == '1'`). Vậy có ít nhất 2 đoạn.

> **Insight:** Với `s[0] == '1'`: `s` có ≥ 2 đoạn 1 **⇔** `"01"` xuất hiện trong `s`.

Nói cách khác, chuỗi hợp lệ phải có dạng `1...10...0` (một khối 1, sau đó toàn 0).

In [8]:
# Kiểm tra insight bằng tay trên vài chuỗi
for s in ["1001", "110", "1", "1110", "1011", "10"]:
    print(f"{s!r:8} chứa '01'? {'01' in s!s:5} | số đoạn = {count_one_segments(s)}")

'1001'   chứa '01'? True  | số đoạn = 2
'110'    chứa '01'? False | số đoạn = 1
'1'      chứa '01'? False | số đoạn = 1
'1110'   chứa '01'? False | số đoạn = 1
'1011'   chứa '01'? True  | số đoạn = 2
'10'     chứa '01'? False | số đoạn = 1


### 4.1. Vì sao điều kiện `s[0] == '1'` quan trọng?

Nếu đề cho phép số 0 ở đầu, ví dụ `"0110"`: chứa `"01"` nhưng chỉ có **1** đoạn 1. Insight sẽ sai! Đây là **phản ví dụ** cho thấy ta phải đọc kỹ constraints.

In [9]:
s = "0110"   # KHÔNG hợp lệ theo đề (có 0 ở đầu), chỉ để minh họa
print("chứa '01':", "01" in s, "| số đoạn thật:", count_one_segments(s))
print("=> insight chỉ đúng khi s[0] == '1'")

chứa '01': True | số đoạn thật: 1
=> insight chỉ đúng khi s[0] == '1'


## 5. Lời giải chính — duyệt một lượt với cờ `seen_zero`

Trước khi dùng mẹo `"01" in s` (rất ngắn nhưng "giấu" logic), ta viết một lời giải duyệt rõ ràng, áp dụng đúng insight:

**Bước 1.** Tạo cờ `seen_zero = False` — "đã gặp số 0 sau đoạn 1 đầu chưa?".

**Bước 2.** Duyệt từng ký tự `ch`:
- Nếu `ch == '0'`: bật `seen_zero = True` (đoạn 1 đầu đã kết thúc).
- Nếu `ch == '1'` mà `seen_zero` đang `True`: đây là đoạn 1 thứ hai → trả `False` ngay.

**Bước 3.** Duyệt hết không vấp → `True`.

In [10]:
# Đoạn 1: cờ boolean bật/tắt
seen_zero = False
print("ban đầu:", seen_zero)
seen_zero = True
print("sau khi gặp '0':", seen_zero)
print("not seen_zero:", not seen_zero)

ban đầu: False
sau khi gặp '0': True
not seen_zero: False


In [11]:
# Đoạn 2 + 3: ghép thành hàm hoàn chỉnh
def check_ones_segment_flag(s: str) -> bool:
    seen_zero = False
    for ch in s:
        if ch == "0":
            seen_zero = True
        elif seen_zero:      # ch == '1' và đã gặp 0 trước đó
            return False
    return True

for s, expected in examples:
    print(s, "->", check_ones_segment_flag(s), "| kỳ vọng", expected)

1001 -> False | kỳ vọng False
110 -> True | kỳ vọng True


Đọc kỹ `elif seen_zero:` — ở nhánh này chắc chắn `ch == '1'` (vì chỉ có 2 loại ký tự và nhánh `if` đã bắt `'0'`). Nên điều kiện đầy đủ là "`'1'` xuất hiện sau một `'0'`", chính là cặp `0...1` → có đoạn thứ hai.

Chú ý: `seen_zero` **không bao giờ tắt lại**. Một khi đã gặp `0`, mọi `'1'` về sau đều là vi phạm.

## 6. TRACE từng bước

Trace `s = "1001"`:

| i | ch | seen_zero trước | Hành động | seen_zero sau |
|---|---|---|---|---|
| 0 | `1` | False | chưa gặp 0 → đoạn đầu, đi tiếp | False |
| 1 | `0` | False | bật cờ | True |
| 2 | `0` | True | vẫn là 0, giữ cờ | True |
| 3 | `1` | True | `'1'` sau `'0'` → **return False** | — |

Trace `s = "110"`: hai `'1'` đầu khi cờ còn False; `'0'` bật cờ; hết chuỗi → `True`.

In [12]:
def trace_check_ones_segment(s: str) -> bool:
    print(f"=== trace s = {s!r} ===")
    seen_zero = False
    for i, ch in enumerate(s):
        before = seen_zero
        if ch == "0":
            seen_zero = True
            action = "gặp 0 -> bật cờ"
        elif seen_zero:
            print(f"i={i}, ch={ch}, seen_zero={before} -> '1' sau '0': return False")
            return False
        else:
            action = "1 thuộc đoạn đầu"
        print(f"i={i}, ch={ch}, seen_zero: {before} -> {seen_zero} | {action}")
    print("Hết chuỗi, không thấy đoạn thứ hai -> return True")
    return True

trace_check_ones_segment("1001")
print()
trace_check_ones_segment("110")

=== trace s = '1001' ===
i=0, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=1, ch=0, seen_zero: False -> True | gặp 0 -> bật cờ
i=2, ch=0, seen_zero: True -> True | gặp 0 -> bật cờ
i=3, ch=1, seen_zero=True -> '1' sau '0': return False

=== trace s = '110' ===
i=0, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=1, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=2, ch=0, seen_zero: False -> True | gặp 0 -> bật cờ
Hết chuỗi, không thấy đoạn thứ hai -> return True


True

### 6.1. Trace nhánh đặc biệt: chuỗi toàn `1` và chuỗi một ký tự

Với `"1111"` cờ không bao giờ bật. Với `"1"` vòng lặp chạy 1 lần. Cả hai đều `True`.

In [13]:
trace_check_ones_segment("1111")
print()
trace_check_ones_segment("1")
print()
trace_check_ones_segment("1110101")

=== trace s = '1111' ===
i=0, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=1, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=2, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=3, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
Hết chuỗi, không thấy đoạn thứ hai -> return True

=== trace s = '1' ===
i=0, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
Hết chuỗi, không thấy đoạn thứ hai -> return True

=== trace s = '1110101' ===
i=0, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=1, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=2, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=3, ch=0, seen_zero: False -> True | gặp 0 -> bật cờ
i=4, ch=1, seen_zero=True -> '1' sau '0': return False


False

## 7. Các lời giải khác (từ cơ bản tới gọn nhất)

### Cách A — Đếm số đoạn (baseline ở mục 3)

### Cách B — `split("0")` rồi đếm mảnh không rỗng

Mỗi mảnh không rỗng sau khi cắt theo `'0'` là một đoạn 1.

In [14]:
def check_ones_segment_split(s: str) -> bool:
    pieces = [p for p in s.split("0") if p]
    return len(pieces) <= 1

for s in ["1001", "110", "1", "1011"]:
    print(s, s.split("0"), "->", check_ones_segment_split(s))

1001 ['1', '', '1'] -> False
110 ['11', ''] -> True
1 ['1'] -> True
1011 ['1', '11'] -> False


### Cách C — Tìm chuỗi con `"01"` (dùng insight trực tiếp)

```python
return "01" not in s
```

Một dòng! Nhưng chỉ đúng vì `s[0] == '1'`.

### Cách D — `rstrip("0")` rồi kiểm tra còn `0` không

`s.rstrip("0")` xóa các số 0 ở **cuối**. Chuỗi hợp lệ `1...10...0` sẽ thành `1...1` (không còn `0`). Nếu sau khi xóa vẫn còn `'0'` thì có `0` nằm **giữa** hai số 1 → hai đoạn.

In [15]:
def check_ones_segment_find(s: str) -> bool:
    return "01" not in s

def check_ones_segment_rstrip(s: str) -> bool:
    return "0" not in s.rstrip("0")

for s in ["1001", "110", "1", "1011", "111000"]:
    print(f"{s!r:9} rstrip -> {s.rstrip('0')!r:7} | find: {check_ones_segment_find(s)} | rstrip: {check_ones_segment_rstrip(s)}")

'1001'    rstrip -> '1001'  | find: False | rstrip: False
'110'     rstrip -> '11'    | find: True | rstrip: True
'1'       rstrip -> '1'     | find: True | rstrip: True
'1011'    rstrip -> '1011'  | find: False | rstrip: False
'111000'  rstrip -> '111'   | find: True | rstrip: True


### 7.1. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| A. Đếm đoạn | đếm điểm bắt đầu `'1'` sau `'0'`/đầu chuỗi | `O(n)` | `O(1)` | tổng quát nhất, đếm được số đoạn |
| B. `split` | mảnh không rỗng = đoạn 1 | `O(n)` | `O(n)` | dễ nghĩ, tốn list phụ |
| Cờ `seen_zero` | `'1'` sau `'0'` là vi phạm | `O(n)` | `O(1)` | dừng sớm, rõ logic |
| C. `"01" in s` | insight | `O(n)` | `O(1)` | gọn nhất, phụ thuộc `s[0]=='1'` |
| D. `rstrip` | xóa 0 cuối, còn 0 là sai | `O(n)` | `O(n)` | tạo chuỗi mới |

`n = len(s)`.

## 8. Code nộp LeetCode

Bản chính (cờ trạng thái, dễ giải thích khi phỏng vấn):

In [16]:
class Solution:
    def checkOnesSegment(self, s: str) -> bool:
        seen_zero = False
        for ch in s:
            if ch == "0":
                seen_zero = True
            elif seen_zero:
                return False
        return True

sol = Solution()
print(sol.checkOnesSegment("1001"))  # False
print(sol.checkOnesSegment("110"))   # True

False
True


Bản một dòng (khi đã hiểu insight):

In [17]:
class SolutionOneLine:
    def checkOnesSegment(self, s: str) -> bool:
        return "01" not in s

print(SolutionOneLine().checkOnesSegment("1001"), SolutionOneLine().checkOnesSegment("110"))

False True


## 9. Vì sao thuật toán đúng?

**Invariant của vòng lặp cờ:** trước khi xử lý ký tự thứ `i`, `seen_zero == True` **khi và chỉ khi** trong `s[0..i-1]` đã có ít nhất một `'0'`.

- Nếu trả `False` tại `i`: `s[i] == '1'` và có `'0'` ở vị trí `j < i`. Vì `s[0] == '1'`, ta có `1 ... 0 ... 1` → ít nhất 2 đoạn 1. Trả `False` là đúng.
- Nếu duyệt hết mà không trả `False`: mọi `'1'` đều đứng trước mọi `'0'`, chuỗi có dạng `1^a 0^b` với `a >= 1` → đúng 1 đoạn. Trả `True` là đúng.

Không bỏ sót: đoạn thứ hai (nếu có) bắt đầu bằng một `'1'` đứng sau `'0'`, vòng lặp chắc chắn đi qua ký tự đó.

## 10. Độ phức tạp

Gọi `n = len(s)`.

- **Time:** `O(n)` — mỗi ký tự xem đúng một lần (có thể dừng sớm).
- **Space:** `O(1)` — chỉ một biến boolean.

Cách `"01" in s` cũng `O(n)` (Python tìm chuỗi con độ dài 2 trên chuỗi dài `n`), bộ nhớ `O(1)`. Cách `split` dùng `O(n)` bộ nhớ cho list.

## 11. Bộ test

Gồm: ví dụ đề, biên (1 ký tự, toàn 1, 1 rồi toàn 0), phản ví dụ dễ sai (`"1101"`, `"10001"`), và đối chiếu mọi cách trên **mọi** chuỗi hợp lệ độ dài ≤ 10.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.checkOnesSegment("1001"), False)
check("đề 2", sol.checkOnesSegment("110"), True)
check("1 ký tự", sol.checkOnesSegment("1"), True)
check("toàn 1", sol.checkOnesSegment("11111"), True)
check("1 rồi toàn 0", sol.checkOnesSegment("1000000"), True)
check("đoạn 2 ở cuối", sol.checkOnesSegment("1101"), False)
check("nhiều 0 giữa", sol.checkOnesSegment("10001"), False)
check("xen kẽ", sol.checkOnesSegment("10101"), False)
check("10", sol.checkOnesSegment("10"), True)

PASS đề 1: False
PASS đề 2: True
PASS 1 ký tự: True
PASS toàn 1: True
PASS 1 rồi toàn 0: True
PASS đoạn 2 ở cuối: False
PASS nhiều 0 giữa: False
PASS xen kẽ: False
PASS 10: True


In [19]:
from itertools import product

solvers = [check_ones_segment_count, check_ones_segment_flag, check_ones_segment_split,
           check_ones_segment_find, check_ones_segment_rstrip,
           Solution().checkOnesSegment, SolutionOneLine().checkOnesSegment]

total = 0
for length in range(1, 11):
    for tail in product("01", repeat=length - 1):
        s = "1" + "".join(tail)          # luôn bắt đầu bằng '1' như đề
        answers = {f(s) for f in solvers}
        assert len(answers) == 1, (s, [f(s) for f in solvers])
        total += 1
print("Tất cả", len(solvers), "cách cho cùng kết quả trên", total, "chuỗi hợp lệ")

Tất cả 7 cách cho cùng kết quả trên 1023 chuỗi hợp lệ


`itertools.product("01", repeat=k)` sinh **mọi** tổ hợp độ dài `k` từ `'0'`, `'1'` — cách kiểm thử vét cạn rất hữu ích khi miền input nhỏ.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `s[i-1]` khi `i == 0`

```python
def wrong(s):
    count = 0
    for i in range(len(s)):
        if s[i] == "1" and s[i - 1] == "0":   # i=0 -> s[-1] là ký tự CUỐI!
            count += 1
    return count <= 1
```

Với `"1001"`: tại `i = 0`, `s[-1]` là `'1'` (ký tự cuối) nên đoạn đầu **bị bỏ sót**; chỉ đếm được 1 đoạn → hàm trả `True`, trong khi đáp án đúng là `False`. Với `"10"` thì lại "vô tình đúng" vì ký tự cuối là `'0'`. Lỗi kiểu này rất khó phát hiện nếu chỉ test vài ví dụ đẹp.

In [20]:
def wrong_count(s):
    count = 0
    for i in range(len(s)):
        if s[i] == "1" and s[i - 1] == "0":
            count += 1
    return count

for s in ["10", "11", "1001"]:
    print(s, "đếm sai:", wrong_count(s), "| đếm đúng:", count_one_segments(s))

10 đếm sai: 1 | đếm đúng: 1
11 đếm sai: 0 | đếm đúng: 1
1001 đếm sai: 1 | đếm đúng: 2


### Lỗi 2: Kiểm tra `"10"` thay vì `"01"`

`"10" in s` chỉ nói "đoạn 1 có kết thúc", không nói "có đoạn thứ hai". `"110"` chứa `"10"` nhưng vẫn hợp lệ.

### Lỗi 3: Đếm số ký tự `'1'` thay vì số đoạn

`s.count("1") <= 1` sai với `"110"` (2 ký tự 1 nhưng 1 đoạn).

In [21]:
def wrong_10(s):  return "10" not in s
def wrong_cnt(s): return s.count("1") <= 1

for s in ["110", "1001"]:
    print(s, "| sai '10':", wrong_10(s), "| sai count:", wrong_cnt(s), "| đúng:", check_ones_segment_flag(s))

110 | sai '10': False | sai count: False | đúng: True
1001 | sai '10': False | sai count: False | đúng: False


### Lỗi 4: Tắt cờ khi gặp lại `'1'`

Nếu viết `if ch == '1': seen_zero = False`, cờ bị reset và ta không còn phát hiện được đoạn thứ hai. Cờ "đã gặp 0" chỉ được bật, không được tắt.

**Sửa:** dùng đúng cấu trúc `if ch == '0': seen_zero = True` / `elif seen_zero: return False`.

## 13. Mẹo nhận dạng pattern

- Đề nói "**đoạn liên tiếp / contiguous segment / group**" → nghĩ tới **điểm bắt đầu đoạn**: phần tử thỏa điều kiện mà phần tử trước không thỏa.
- Hỏi "**nhiều nhất 1 đoạn**" → tương đương "không có điểm chuyển `0 → 1` thứ hai" → thường là kiểm tra một mẫu (pattern) như `"01"`.
- Chuỗi nhị phân + câu hỏi về hình dạng (`1...10...0`) → thử `rstrip`/`lstrip`/`split`.
- Luôn đọc constraint đặc biệt như `s[0] == '1'` — nó thường là chìa khóa của lời giải ngắn.

## 14. Biến thể

### 14.1. Đếm số đoạn 1 và độ dài đoạn dài nhất

Dùng lại kỹ thuật "điểm bắt đầu đoạn", thêm biến `run` đếm độ dài đoạn hiện tại.

In [22]:
def segments_info(s: str):
    count, run, best = 0, 0, 0
    for ch in s:
        if ch == "1":
            if run == 0:
                count += 1      # bắt đầu đoạn mới
            run += 1
            best = max(best, run)
        else:
            run = 0
    return count, best

for s in ["1001", "1110110", "0", "0110"]:
    print(s, "-> (số đoạn, dài nhất) =", segments_info(s))

1001 -> (số đoạn, dài nhất) = (2, 1)
1110110 -> (số đoạn, dài nhất) = (2, 3)
0 -> (số đoạn, dài nhất) = (0, 0)
0110 -> (số đoạn, dài nhất) = (1, 2)


### 14.2. Bỏ điều kiện `s[0] == '1'` (cho phép 0 ở đầu)

Khi đó cần bỏ các số 0 ở đầu trước: `"01" not in s.lstrip("0")`. Hoặc đếm đoạn trực tiếp như baseline (luôn đúng).

In [23]:
def at_most_one_segment_general(s: str) -> bool:
    return "01" not in s.lstrip("0")

for s in ["0110", "0101", "000", "1001"]:
    print(s, "->", at_most_one_segment_general(s), "| đếm đoạn:", count_one_segments(s))

0110 -> True | đếm đoạn: 1
0101 -> False | đếm đoạn: 2
000 -> True | đếm đoạn: 0
1001 -> False | đếm đoạn: 2


### 14.3. Liên hệ: LeetCode 485 (Max Consecutive Ones), 1869 (Longer Contiguous Segments of Ones than Zeros), 1446 (Consecutive Characters) — đều dùng "biến đếm độ dài đoạn hiện tại".

## 15. Bài tập luyện tập (tăng dần)

**Bài 1 (đoán output):** Không chạy code, hãy đoán `"1011000".split("0")` và `"1011000".rstrip("0")`.

**Bài 2 (tính tay):** Lập bảng trace cờ `seen_zero` cho `s = "1110100"`. Hàm trả gì, tại chỉ số nào?

**Bài 3 (điền TODO):** Hoàn thiện hàm đếm số đoạn 1.

In [24]:
def ex3_count_segments(s: str) -> int:
    # TODO 1: khởi tạo count = 0
    # TODO 2: duyệt i trong range(len(s))
    # TODO 3: nếu s[i] == '1' và (i == 0 hoặc s[i-1] == '0') thì count += 1
    pass

# Tự làm xong rồi mới mở test:
# assert ex3_count_segments("1001") == 2
# assert ex3_count_segments("0110") == 1

**Bài 4 (tự viết):** Viết `ex4_at_most_one_zero_segment(s)` — kiểm tra chuỗi nhị phân (bất kỳ) có nhiều nhất một đoạn `0` liên tiếp không.

In [25]:
def ex4_at_most_one_zero_segment(s: str) -> bool:
    # TODO: tự viết
    pass

# assert ex4_at_most_one_zero_segment("1001") == True
# assert ex4_at_most_one_zero_segment("0100") == False
# assert ex4_at_most_one_zero_segment("111") == True

**Bài 5 (biến thể):** Viết `ex5_at_most_k_segments(s, k)` — trả `True` nếu số đoạn 1 `<= k`. Dừng sớm khi vượt `k`.

In [26]:
def ex5_at_most_k_segments(s: str, k: int) -> bool:
    # TODO: đếm điểm bắt đầu đoạn, vượt k thì return False
    pass

# assert ex5_at_most_k_segments("10101", 3) == True
# assert ex5_at_most_k_segments("10101", 2) == False
# assert ex5_at_most_k_segments("000", 0) == True

**Bài 6 (mini challenge):** Cho chuỗi nhị phân `s` bắt đầu bằng `'1'`. Tìm **số ký tự ít nhất cần đổi** (`0↔1`) để `s` có đúng một đoạn 1 bắt đầu từ đầu chuỗi (dạng `1^a 0^b`, `a >= 1`). Gợi ý: thử mọi điểm cắt `a`.

In [27]:
def ex6_min_flips(s: str) -> int:
    # TODO: thử mọi a từ 1..len(s); chi phí = số '0' trong s[:a] + số '1' trong s[a:]
    pass

# assert ex6_min_flips("1001") == 1
# assert ex6_min_flips("110") == 0
# assert ex6_min_flips("10101") == 2

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | `split` cắt tại mỗi `0`; `rstrip` chỉ xóa ở cuối | đếm số `0` liền nhau | có mảnh rỗng `''` |
| 2 | cờ chỉ bật khi gặp `0` | xem `'1'` đầu tiên sau `'0'` | chỉ số 4 |
| 3 | điểm bắt đầu đoạn | `i == 0 or s[i-1] == '0'` | `if s[i]=='1' and (...): count += 1` |
| 4 | đối xứng bài gốc, đổi vai 0/1 | đếm điểm bắt đầu đoạn 0 | `count <= 1` |
| 5 | giống bài 3 | thêm `if count > k: return False` | return True cuối hàm |
| 6 | prefix 1, suffix 0 | đếm theo từng `a` | `s[:a].count('0') + s[a:].count('1')` |

## 17. Đáp án tham khảo

**Bài 1:** `"1011000".split("0")` → `['1', '11', '', '', '']`; `rstrip("0")` → `'1011'` (vẫn còn `0` ở giữa → 2 đoạn).

**Bài 2:** `i=0,1,2` là `1` (cờ False), `i=3` là `0` → bật cờ, `i=4` là `1` khi cờ True → **return False tại i = 4**.

In [28]:
print("1011000".split("0"), "1011000".rstrip("0"))

def ex3_count_segments_ans(s: str) -> int:
    count = 0
    for i in range(len(s)):
        if s[i] == "1" and (i == 0 or s[i - 1] == "0"):
            count += 1
    return count

def ex4_at_most_one_zero_segment_ans(s: str) -> bool:
    count = 0
    for i in range(len(s)):
        if s[i] == "0" and (i == 0 or s[i - 1] == "1"):
            count += 1
    return count <= 1

def ex5_at_most_k_segments_ans(s: str, k: int) -> bool:
    count = 0
    for i in range(len(s)):
        if s[i] == "1" and (i == 0 or s[i - 1] == "0"):
            count += 1
            if count > k:
                return False
    return True

def ex6_min_flips_ans(s: str) -> int:
    best = len(s)
    for a in range(1, len(s) + 1):
        cost = s[:a].count("0") + s[a:].count("1")
        best = min(best, cost)
    return best

assert trace_check_ones_segment("1110100") == False
assert ex3_count_segments_ans("1001") == 2
assert ex3_count_segments_ans("0110") == 1
assert ex4_at_most_one_zero_segment_ans("1001") == True
assert ex4_at_most_one_zero_segment_ans("0100") == False
assert ex4_at_most_one_zero_segment_ans("111") == True
assert ex5_at_most_k_segments_ans("10101", 3) == True
assert ex5_at_most_k_segments_ans("10101", 2) == False
assert ex5_at_most_k_segments_ans("000", 0) == True
assert ex6_min_flips_ans("1001") == 1
assert ex6_min_flips_ans("110") == 0
assert ex6_min_flips_ans("10101") == 2
print("Đáp án tham khảo PASS toàn bộ test!")

['1', '11', '', '', ''] 1011
=== trace s = '1110100' ===
i=0, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=1, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=2, ch=1, seen_zero: False -> False | 1 thuộc đoạn đầu
i=3, ch=0, seen_zero: False -> True | gặp 0 -> bật cờ
i=4, ch=1, seen_zero=True -> '1' sau '0': return False
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def checkOnesSegment(self, s: str) -> bool:
        seen_zero = False
        for ch in s:
            if ch == "0":
                seen_zero = True
            elif seen_zero:        # '1' xuất hiện sau '0' -> đoạn thứ hai
                return False
        return True
        # hoặc ngắn gọn: return "01" not in s   (vì s[0] == '1')
```

**Template "điểm bắt đầu đoạn":** `s[i] thỏa điều kiện and (i == 0 or s[i-1] không thỏa)`.

**Checklist trước khi submit:**

- [ ] Đã đọc constraint `s[0] == '1'` và hiểu vì sao `"01"` là đủ?
- [ ] Không dùng `s[i-1]` khi `i == 0`?
- [ ] Đếm **đoạn**, không đếm **ký tự** `'1'`?
- [ ] Test chuỗi 1 ký tự `"1"`, toàn `1`, `1` rồi toàn `0`, và `"1101"`?
- [ ] Tên method đúng `checkOnesSegment`, trả về `bool`?